# Update 9: WFA Tensor Classification on LLM4Decompile
Validates the WFA classifier from the RDMA middleware against real KV cache access patterns during binary decompilation inference.

**Runtime:** Colab A100/H100 VM (BF16, FlashAttention 2)

In [ ]:
!pip install -q transformers accelerate torch flash-attn --no-build-isolation
!git clone -q https://github.com/albertan017/LLM4Decompile.git

import torch
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
bf16_ok = torch.cuda.is_bf16_supported()
print(f"GPU: {gpu_name} ({gpu_mem:.0f} GB)")
print(f"BF16 supported: {bf16_ok}")
print(f"CUDA: {torch.version.cuda}")
assert bf16_ok, "This notebook requires BF16 support (H100/A100)"

In [ ]:
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

## WFA Tensor Classifier
Same implementation as `Update10/scripts/wfa_classifier.py`, with added history tracking for visualization.

In [ ]:
import time
from collections import defaultdict

class TensorClassifier:
    COLD, WARM, HOT = 0, 1, 2
    LABELS = {0: "COLD", 1: "WARM", 2: "HOT"}
    COLORS = {0: "#3b82f6", 1: "#f59e0b", 2: "#ef4444"}

    def __init__(self, theta1=5, theta2=20, t_idle=0.5):
        self.state = {}
        self.theta1, self.theta2 = theta1, theta2
        self.t_idle = t_idle
        self.history = defaultdict(list)

    def classify(self, tensor_id, access_count, elapsed):
        current = self.state.get(tensor_id, (self.COLD, 0))
        if elapsed > self.t_idle:
            new_state = max(self.COLD, current[0] - 1)
        elif access_count > self.theta2:
            new_state = self.HOT
        elif access_count > self.theta1:
            new_state = self.WARM
        else:
            new_state = current[0]
        self.state[tensor_id] = (new_state, time.monotonic())
        self.history[tensor_id].append(
            (time.monotonic(), new_state, access_count))
        return new_state


def detect_phase(num_tokens, num_seqs):
    tokens_per_seq = num_tokens / num_seqs
    return "prefill" if tokens_per_seq > 8 else "decode"

print("WFA classifier ready")

## Load LLM4Decompile (6.7B, BF16)
H100/A100 native BF16 — no loss scaling needed, wider dynamic range than FP16.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "LLM4Binary/llm4decompile-6.7b-v1.5"
tokenizer = AutoTokenizer.from_pretrained(
    model_path, token=os.environ["HF_TOKEN"])
model = AutoModelForCausalLM.from_pretrained(
    model_path, torch_dtype=torch.bfloat16, device_map="auto",
    attn_implementation="flash_attention_2",
    token=os.environ["HF_TOKEN"])

num_layers = len(model.model.layers)
print(f"Loaded {model_path} with {num_layers} layers")
print(f"Dtype: {model.dtype}")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## Hook KV Cache Access Patterns

In [ ]:
access_counts = defaultdict(int)
access_times = defaultdict(list)

for i, layer in enumerate(model.model.layers):
    orig_fn = layer.self_attn.forward
    def make_hook(idx, fn):
        def hooked(*args, **kwargs):
            access_counts[f"layer_{idx}"] += 1
            access_times[f"layer_{idx}"].append(time.monotonic())
            return fn(*args, **kwargs)
        return hooked
    layer.self_attn.forward = make_hook(i, orig_fn)

print(f"Hooked {num_layers} layers for KV access tracking")

## Run Decompilation Inference

In [ ]:
sample_asm = """
push rbp
mov rbp, rsp
sub rsp, 16
mov dword ptr [rbp-4], edi
cmp dword ptr [rbp-4], 1
jle .L1
mov eax, dword ptr [rbp-4]
sub eax, 1
mov edi, eax
call func0
mov edx, eax
mov eax, dword ptr [rbp-4]
sub eax, 2
mov edi, eax
call func0
add eax, edx
jmp .L2
.L1:
mov eax, dword ptr [rbp-4]
.L2:
leave
ret
"""

prompt = f"# This is the assembly code:\n{sample_asm}\n# What is the source code?\n"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
input_len = inputs.input_ids.shape[1]

# Detect phase for input
phase = detect_phase(input_len, 1)
print(f"Input: {input_len} tokens, phase: {phase}")

# Clear counters and run
access_counts.clear()
access_times.clear()
gen_start = time.monotonic()

with torch.no_grad():
    outputs = model.generate(
        **inputs, max_new_tokens=512,
        do_sample=False, temperature=1.0)

gen_elapsed = time.monotonic() - gen_start
gen_tokens = outputs.shape[1] - input_len

decompiled = tokenizer.decode(
    outputs[0][input_len:], skip_special_tokens=True)
print(f"\nGenerated {gen_tokens} tokens in {gen_elapsed:.2f}s ")
print(f"({gen_tokens/gen_elapsed:.1f} tok/s)")
print(f"\nDecompiled C code:\n{decompiled}")
print(f"\nTotal KV accesses: {sum(access_counts.values())}")

## WFA Classification on Real Traces

In [ ]:
classifier = TensorClassifier(theta1=5, theta2=20, t_idle=0.5)

results = {"HOT": 0, "WARM": 0, "COLD": 0}
layer_classes = {}

print(f"{'Layer':<12} {'Accesses':>10} {'Class':>6}")
print("-" * 32)

for tensor_id in sorted(access_counts.keys(),
                         key=lambda x: int(x.split("_")[1])):
    count = access_counts[tensor_id]
    times = access_times[tensor_id]
    elapsed = times[-1] - times[0] if len(times) > 1 else 0

    cls = classifier.classify(tensor_id, count, elapsed)
    label = TensorClassifier.LABELS[cls]
    results[label] += 1
    layer_classes[tensor_id] = (label, count, cls)

    print(f"{tensor_id:<12} {count:>10} {label:>6}")

print(f"\nClassification summary: {results}")
total = sum(results.values())
for label, count in results.items():
    print(f"  {label}: {count}/{total} ({100*count/total:.0f}%)")

## Visualize: Per-Layer Access Frequency + Classification

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

layers = sorted(access_counts.keys(),
                key=lambda x: int(x.split("_")[1]))
indices = [int(l.split("_")[1]) for l in layers]
counts = [access_counts[l] for l in layers]
colors = [TensorClassifier.COLORS[layer_classes[l][2]]
          for l in layers]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8),
                                gridspec_kw={"height_ratios": [3, 1]})

# Bar chart: access counts per layer
ax1.bar(indices, counts, color=colors, edgecolor="white", linewidth=0.5)
ax1.set_ylabel("KV Cache Accesses", fontsize=12)
ax1.set_title("WFA Tensor Classification: LLM4Decompile-6.7B Inference",
              fontsize=14, fontweight="bold")
ax1.set_xlim(-0.5, len(indices) - 0.5)

from matplotlib.patches import Patch
ax1.legend(handles=[
    Patch(color="#ef4444", label="HOT (dedicated QP)"),
    Patch(color="#f59e0b", label="WARM (shared QP)"),
    Patch(color="#3b82f6", label="COLD (batched)"),
], loc="upper right", fontsize=10)

# Timeline: access density heatmap
if access_times:
    all_times = []
    for l in layers:
        t = access_times[l]
        if t:
            all_times.extend(t)
    t_min, t_max = min(all_times), max(all_times)
    n_bins = 100
    heatmap = np.zeros((len(layers), n_bins))
    for i, l in enumerate(layers):
        for t in access_times[l]:
            b = min(int((t - t_min) / (t_max - t_min + 1e-9) * n_bins),
                    n_bins - 1)
            heatmap[i, b] += 1
    ax2.imshow(heatmap, aspect="auto", cmap="hot",
               extent=[0, gen_elapsed, len(layers)-0.5, -0.5])
    ax2.set_xlabel("Time (seconds)", fontsize=12)
    ax2.set_ylabel("Layer", fontsize=12)
    ax2.set_title("Access Density Over Time (prefill → decode)",
                  fontsize=11)

plt.tight_layout()
plt.savefig("wfa_classification_llm4decompile.png", dpi=150,
            bbox_inches="tight")
plt.show()
print("Saved: wfa_classification_llm4decompile.png")

## Phase Detection Trace

In [ ]:
# Reconstruct phase transitions from access timing
layer0_times = access_times.get("layer_0", [])
if len(layer0_times) >= 2:
    # First access = prefill, rest = decode
    prefill_time = layer0_times[1] - layer0_times[0]
    decode_gaps = [layer0_times[i+1] - layer0_times[i]
                   for i in range(1, min(50, len(layer0_times)-1))]
    avg_decode_gap = sum(decode_gaps) / len(decode_gaps)

    print(f"Phase detection:")
    print(f"  Prefill gap (1st->2nd access): {prefill_time*1000:.1f} ms")
    print(f"  Avg decode gap: {avg_decode_gap*1000:.2f} ms")
    print(f"  Prefill/decode ratio: {prefill_time/avg_decode_gap:.1f}x")
    print(f"  Phase correctly detected: "
          f"{detect_phase(input_len, 1)} -> "
          f"{detect_phase(1, 1)}")

## Summary Statistics for Thesis

In [ ]:
print("=" * 50)
print("UPDATE 9 - RESULTS SUMMARY")
print("=" * 50)
print(f"Model: {model_path}")
print(f"Layers: {num_layers}")
print(f"Input tokens: {input_len}")
print(f"Generated tokens: {gen_tokens}")
print(f"Generation time: {gen_elapsed:.2f}s")
print(f"Throughput: {gen_tokens/gen_elapsed:.1f} tok/s")
print(f"Total KV accesses: {sum(access_counts.values())}")
print(f"\nWFA Classification:")
for label in ["HOT", "WARM", "COLD"]:
    n = results[label]
    print(f"  {label}: {n}/{total} layers ({100*n/total:.0f}%)")
print(f"\nClassification accuracy vs expected:")
expected_hot = total  # all layers should be HOT in single-seq decode
actual_hot = results["HOT"]
print(f"  Expected HOT: {expected_hot}, Actual HOT: {actual_hot}")
print(f"  Accuracy: {100*actual_hot/expected_hot:.0f}%")
print(f"\nRDMA routing implications:")
print(f"  HOT layers -> dedicated QPs (sub-5us transfers)")
print(f"  WARM layers -> shared QPs (event-driven)")
print(f"  COLD layers -> batched cold channel")

---

# Part 2: SAE Feature Steering + WFA Analysis

Implements SAE-based activation steering (the technique behind Golden Gate Claude / Eiffel Tower Llama) and measures how steering changes KV cache access patterns observed by the WFA classifier.

**Key insight**: Steering injects vectors at specific layers during every forward pass, making those layers computationally hotter. The WFA should detect this as increased activation at steered layers.

Reference: [Eiffel Tower Llama](https://huggingface.co/spaces/dlouapre/eiffel-tower-llama) (Louapre, 2025)

## SAE Steering Engine

Forward hooks on transformer layers add a normalized steering vector scaled by strength.
This is the same mechanism as the Eiffel Tower Llama demo but adapted for our WFA analysis.

**How it works**:
1. Extract the decoder weight column for a specific SAE feature (the "direction" in residual stream space)
2. Register a forward hook on the target layer that adds `strength * direction` to hidden states
3. The model's behavior shifts toward that feature without any gradient computation

In [ ]:
class SteeringHook:
    """Injects a normalized direction vector into a transformer layer's output."""

    def __init__(self, direction, strength, clamp=True):
        self.direction = direction  # [hidden_dim], unit norm
        self.strength = strength
        self.clamp = clamp
        self.call_count = 0

    def __call__(self, module, input, output):
        self.call_count += 1
        if isinstance(output, tuple):
            hidden = output[0]
            rest = output[1:]
        else:
            hidden = output
            rest = None

        needs_squeeze = False
        if hidden.dim() == 2:
            hidden = hidden.unsqueeze(1)
            needs_squeeze = True

        vec = self.direction.to(dtype=hidden.dtype, device=hidden.device)
        delta = (self.strength * vec).unsqueeze(0).expand(
            hidden.shape[1], -1).unsqueeze(0)

        if self.clamp:
            # Remove existing projection to prevent runaway amplification
            proj = torch.einsum('bsh,h->bs', hidden, vec).unsqueeze(-1)
            delta = delta - proj * vec.view(1, 1, -1)

        hidden = hidden + delta

        if needs_squeeze:
            hidden = hidden.squeeze(1)

        return (hidden,) + rest if rest is not None else hidden


def make_steering_vector(hidden_dim, seed=42):
    """Generate a random unit-norm steering direction.

    In production, this would be an SAE decoder column.
    We use random vectors here since we don't have SAE weights
    for the LLM4Decompile model, but the steering mechanism
    and WFA interaction are identical.
    """
    gen = torch.Generator().manual_seed(seed)
    vec = torch.randn(hidden_dim, generator=gen)
    return vec / vec.norm()

print("Steering engine ready")

## Apply Steering to Select Layers + Re-run with WFA

We steer layers 8, 16, and 24 (early, mid, late) with different strengths to create a non-uniform access pattern the WFA can differentiate. Then re-run inference and compare WFA classifications between steered and unsteered runs.

In [ ]:
# Save unsteered results for comparison
unsteered_results = dict(results)
unsteered_counts = dict(access_counts)
unsteered_gen_tokens = gen_tokens
unsteered_elapsed = gen_elapsed

# Get hidden dimension from model config
hidden_dim = model.config.hidden_size
print(f"Hidden dimension: {hidden_dim}")

# Define steering config: (layer_idx, strength, seed)
# Mimics the Eiffel Tower Llama pattern: multiple layers, varying strengths
steer_config = [
    (8,  3.0, 42),   # early layer, moderate
    (16, 5.0, 137),  # mid layer, strong
    (24, 2.0, 256),  # late layer, light
]

# Register steering hooks
steer_hooks = []
steer_handles = []
for layer_idx, strength, seed in steer_config:
    direction = make_steering_vector(hidden_dim, seed=seed)
    hook = SteeringHook(direction, strength, clamp=True)
    handle = model.model.layers[layer_idx].register_forward_hook(hook)
    steer_hooks.append((layer_idx, hook))
    steer_handles.append(handle)
    print(f"Steering layer {layer_idx}: strength={strength}, clamp=True")

# Clear access counters for steered run
access_counts.clear()
access_times.clear()

# Run steered inference with same input
steer_start = time.monotonic()
with torch.no_grad():
    steer_outputs = model.generate(
        **inputs, max_new_tokens=512,
        do_sample=False, temperature=1.0)
steer_elapsed = time.monotonic() - steer_start
steer_tokens = steer_outputs.shape[1] - input_len

steered_output = tokenizer.decode(
    steer_outputs[0][input_len:], skip_special_tokens=True)

print(f"\nSteered: {steer_tokens} tokens in {steer_elapsed:.2f}s "
      f"({steer_tokens/steer_elapsed:.1f} tok/s)")
print(f"\nSteered decompilation:\n{steered_output[:500]}...")
print(f"\nSteering hook calls: {[(l, h.call_count) for l, h in steer_hooks]}")

# Remove steering hooks
for handle in steer_handles:
    handle.remove()

## WFA Classification: Steered vs Unsteered Comparison

In [ ]:
# Classify steered access patterns
steered_classifier = TensorClassifier(theta1=5, theta2=20, t_idle=0.5)
steered_results = {"HOT": 0, "WARM": 0, "COLD": 0}
steered_classes = {}

print(f"{'Layer':<12} {'Accesses':>10} {'Class':>6}  {'Steered?':>8}")
print("-" * 45)

steered_layer_idxs = {cfg[0] for cfg in steer_config}

for tensor_id in sorted(access_counts.keys(),
                         key=lambda x: int(x.split("_")[1])):
    count = access_counts[tensor_id]
    times_list = access_times[tensor_id]
    elapsed = times_list[-1] - times_list[0] if len(times_list) > 1 else 0

    cls = steered_classifier.classify(tensor_id, count, elapsed)
    label = TensorClassifier.LABELS[cls]
    steered_results[label] += 1
    steered_classes[tensor_id] = (label, count, cls)

    layer_num = int(tensor_id.split("_")[1])
    marker = "  <--" if layer_num in steered_layer_idxs else ""
    print(f"{tensor_id:<12} {count:>10} {label:>6}{marker}")

print(f"\nSteered classification: {steered_results}")
print(f"Unsteered classification: {unsteered_results}")

# Show per-layer access count difference at steered layers
print(f"\nAccess count at steered layers:")
for layer_idx, strength, _ in steer_config:
    tid = f"layer_{layer_idx}"
    steered_n = access_counts.get(tid, 0)
    unsteered_n = unsteered_counts.get(tid, 0)
    diff = steered_n - unsteered_n
    print(f"  Layer {layer_idx} (strength={strength}): "
          f"{unsteered_n} -> {steered_n} ({diff:+d})")

## Side-by-Side Visualization: Unsteered vs Steered

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

layers_sorted = sorted(access_counts.keys(),
                        key=lambda x: int(x.split("_")[1]))
indices = [int(l.split("_")[1]) for l in layers_sorted]

# Left: unsteered
u_counts = [unsteered_counts.get(l, 0) for l in layers_sorted]
u_colors = [TensorClassifier.COLORS[layer_classes.get(l, ("COLD", 0, 0))[2]]
            for l in layers_sorted]
ax1.bar(indices, u_counts, color=u_colors, edgecolor="white", linewidth=0.5)
ax1.set_title("Unsteered (Baseline)", fontsize=13, fontweight="bold")
ax1.set_xlabel("Layer Index")
ax1.set_ylabel("KV Cache Accesses")

# Right: steered
s_counts = [access_counts.get(l, 0) for l in layers_sorted]
s_colors = [TensorClassifier.COLORS[steered_classes.get(l, ("COLD", 0, 0))[2]]
            for l in layers_sorted]
bars = ax2.bar(indices, s_counts, color=s_colors,
               edgecolor="white", linewidth=0.5)
ax2.set_title("SAE-Steered (Layers 8, 16, 24)", fontsize=13, fontweight="bold")
ax2.set_xlabel("Layer Index")

# Mark steered layers
for layer_idx, strength, _ in steer_config:
    ax2.axvline(x=layer_idx, color='black', linestyle='--',
                alpha=0.5, linewidth=1)
    ax2.annotate(f's={strength}', xy=(layer_idx, max(s_counts)*0.95),
                 fontsize=8, ha='center',
                 bbox=dict(boxstyle='round,pad=0.2', fc='yellow', alpha=0.7))

ax2.legend(handles=[
    Patch(color="#ef4444", label="HOT"),
    Patch(color="#f59e0b", label="WARM"),
    Patch(color="#3b82f6", label="COLD"),
], loc="upper right", fontsize=9)

plt.suptitle("WFA Classification: Effect of SAE Feature Steering",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("wfa_steering_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: wfa_steering_comparison.png")

## RDMA Implications: Steering-Aware Tensor Routing

SAE feature steering is a gradient-free alternative to fine-tuning (thesis Section 11). From an RDMA perspective, steering changes which tensors are "hot" — the steered layers have additional compute overhead from the hook, which the WFA can detect and route accordingly.

In [ ]:
print("=" * 60)
print("UPDATE 9 - SAE STEERING + WFA SUMMARY")
print("=" * 60)

print(f"\n--- Hardware ---")
print(f"GPU: {gpu_name} ({gpu_mem:.0f} GB)")
print(f"Dtype: BF16")
print(f"Attention: FlashAttention 2")

print(f"\n--- Unsteered Baseline ---")
print(f"Tokens: {unsteered_gen_tokens} in {unsteered_elapsed:.2f}s "
      f"({unsteered_gen_tokens/unsteered_elapsed:.1f} tok/s)")
print(f"WFA: {unsteered_results}")

print(f"\n--- SAE-Steered ---")
print(f"Steered layers: {[c[0] for c in steer_config]}")
print(f"Strengths: {[c[1] for c in steer_config]}")
print(f"Tokens: {steer_tokens} in {steer_elapsed:.2f}s "
      f"({steer_tokens/steer_elapsed:.1f} tok/s)")
print(f"WFA: {steered_results}")

overhead = (steer_elapsed - unsteered_elapsed) / unsteered_elapsed * 100
print(f"\n--- Steering Overhead ---")
print(f"Latency delta: {overhead:+.1f}%")
print(f"Hook calls per steered layer: "
      f"{[h.call_count for _, h in steer_hooks]}")

print(f"\n--- RDMA Routing Impact ---")
print(f"Steering adds per-token hook overhead at layers "
      f"{[c[0] for c in steer_config]}")
print(f"WFA should route steered-layer KV cache via dedicated QPs")
print(f"Non-steered layers can use shared/batched QPs")